<a href="https://colab.research.google.com/github/riccardogs/PyTorch_tutorial/blob/main/Transforms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transforms


Data does not always come in its final processed form that is required for training machine learning algorithms. We use transforms to perform some manipulation of the data and make it suitable for training.

All TorchVision datasets have two parameters
- transform to modify the features
- target_transform to modify the labels - that accept callables containing the transformation logic.

The torchvision.transforms module offers several commonly-used transforms out of the box.

The FashionMNIST features are in PIL Image format, and the labels are integers. For training, we need the features as normalized tensors, and the labels as one-hot encoded tensors. To make these transformations, we use ToTensor and Lambda.

Alre trasformazioni : https://docs.pytorch.org/vision/stable/transforms.html

In [1]:
import torch
from torchvision import datasets

from torchvision.transforms import ToTensor, Lambda
# from torchvision.transforms import ToTensor, Lambda: importa due trasformazioni
# - ToTensor: converte immagini in tensor e normalizza pixel a [0,1]
# - Lambda: permette di creare trasformazioni personalizzate con funzioni lambda

ds = datasets.FashionMNIST(
    # ds = datasets.FashionMNIST: crea un dataset Fashion-MNIST
    # - È lo stesso dataset di prima, ma con una trasformazione diversa per le etichette

    root="data",
    # root="data": directory dove salvare/scaricare il dataset

    train=True,
    # train=True: carica il set di training (60.000 immagini)

    download=True,
    # download=True: scarica il dataset se non è già presente

    transform=ToTensor(),
    # transform=ToTensor(): trasforma le immagini in tensor
    # - Converte PIL → tensor
    # - Normalizza pixel da [0,255] a [0.0,1.0]
    # - Cambia forma da (H,W) a (C,H,W) con C=1

    target_transform=Lambda(lambda y: torch.zeros(10, dtype=torch.float).scatter_(0, torch.tensor(y), value=1))
    # target_transform=Lambda(...): trasforma l'etichetta in one-hot encoding
    # - Lambda: crea una trasformazione personalizzata
    # - lambda y: funzione lambda che prende l'etichetta originale y (0-9)

    # DENTRO LA LAMBDA:
    # torch.zeros(10, dtype=torch.float): crea un tensor di 10 zeri [0.,0.,0.,...]
    # .scatter_(0, torch.tensor(y), value=1): mette 1 alla posizione dell'etichetta
    #   * 0: dimensione lungo cui fare lo scatter
    #   * torch.tensor(y): indice dove mettere il valore (es. 3)
    #   * value=1: valore da mettere
    #   * _ (underscore): operazione in-place (modifica direttamente il tensor)

    # ESEMPIO: se y=3
    # - torch.zeros(10) → [0.,0.,0.,0.,0.,0.,0.,0.,0.,0.]
    # - .scatter_(0, 3, 1) → [0.,0.,0.,1.,0.,0.,0.,0.,0.,0.]
    # Questo è one-hot encoding: un vettore con un 1 alla posizione della classe
)

# COSA CAMBIA RISPETTO A PRIMA:
# Prima: target_transform non specificato → etichetta era un numero (es. 3)
# Ora: target_transform converte in one-hot → etichetta è un tensor [0.,0.,0.,1.,0.,0.,0.,0.,0.,0.]

# PERCHÉ USARE ONE-HOT ENCODING:
# Molte funzioni di loss in PyTorch (es. CrossEntropyLoss) possono accettare sia:
# - Etichette come interi (sparse labels)
# - Etichette one-hot (probabilità target)
# One-hot è utile per certe architetture o quando si vuole esplicitamente una distribuzione

100%|██████████| 26.4M/26.4M [00:02<00:00, 11.0MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 227kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.46MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 9.52MB/s]


# ToTensor()

ToTensor converts a PIL image or NumPy ndarray into a FloatTensor. and scales the image’s pixel intensity values in the range [0., 1.]


# Lambda Transforms

Lambda transforms apply any user-defined lambda function. Here, we define a function to turn the integer into a one-hot encoded tensor. It first creates a zero tensor of size 10 (the number of labels in our dataset) and calls scatter_ which assigns a value=1 on the index as given by the label y.

In [ ]:
target_transform = Lambda(lambda y: torch.zeros(10, dtype=torch.float).scatter_(dim=0, index=torch.tensor(y), value=1))

# target_transform = Lambda(...): crea una trasformazione per convertire etichette in one-hot encoding

# ANALIZZIAMO DALL'INTERNO VERSO L'ESTERNO:

# 1. torch.zeros(10, dtype=torch.float)
#    - Crea un tensor di 10 elementi tutti zero: [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]
#    - dtype=torch.float: i valori sono float (con punto decimale)

# 2. .scatter_(dim=0, index=torch.tensor(y), value=1)
#    - scatter_() è un metodo che "sparge" un valore in posizioni specifiche
#    - Il _ finale significa che l'operazione è IN-PLACE (modifica il tensor direttamente)
#
#    Parametri:
#    - dim=0: opera sulla prima dimensione (lungo il vettore)
#    - index=torch.tensor(y): l'indice dove mettere il valore (y è l'etichetta 0-9)
#      * Se y=3, index sarà tensor(3)
#    - value=1: il valore da mettere alla posizione specificata

# 3. lambda y: ...
#    - Crea una funzione anonima che prende y (l'etichetta originale) come input
#    - Esegue tutto il codice sopra per convertire y in one-hot

# 4. Lambda(...)
#    - Avvolge la funzione lambda in una trasformazione PyTorch
#    - La rende utilizzabile come parametro target_transform


# QUINDI:
# target_transform trasforma un numero intero (0-9) in un vettore one-hot
# Questo è utile per modelli che prevedono una distribuzione di probabilità su 10 classi